# Employees System Project
A small OOP project made of three classes:

| Class | Responsibility |
|---|---|
| `Employee` | Holds `name`, `age`, `salary`; string representation and formatted output |
| `EmployeesManager` | Add, list, delete-by-age-range, find-by-name, update-salary-by-name |
| `FrontendManager` | Text menu (user interface) that talks to the manager |

The last section runs the menu with **scripted input** so you can see it work in the notebook. To use it for real, run `FrontendManager().run()` in an empty cell and type your choices.

## 1. `Employee` class

In [1]:
class Employee:
    def __init__(self, name, age, salary):
        self.name = name
        self.age = age
        self.salary = salary

    def __str__(self):
        """Human-friendly one-line representation."""
        return f"{self.name:<15} Age: {self.age:<4} Salary: {self.salary:,.2f}"

    def __repr__(self):
        return f"Employee(name={self.name!r}, age={self.age}, salary={self.salary})"

    def to_formatted_string(self):
        """Formatted output used when listing employees."""
        return f"| {self.name:<15} | {self.age:^5} | {self.salary:>12,.2f} |"

e = Employee("Asha", 28, 55000)
print(e)
print(repr(e))
print(e.to_formatted_string())

Asha            Age: 28   Salary: 55,000.00
Employee(name='Asha', age=28, salary=55000)
| Asha            |  28   |    55,000.00 |


## 2. `EmployeesManager` class
Stores employees in a list and offers the five operations from the spec.

In [2]:
class EmployeesManager:
    def __init__(self):
        self.employees = []

    def add_employee(self, employee):
        self.employees.append(employee)

    def list_employees(self):
        if not self.employees:
            print("No employees found.")
            return
        line = "+" + "-" * 17 + "+" + "-" * 7 + "+" + "-" * 14 + "+"
        print(line)
        print(f"| {'Name':<15} | {'Age':^5} | {'Salary':>12} |")
        print(line)
        for emp in self.employees:
            print(emp.to_formatted_string())
        print(line)

    def delete_by_age_range(self, from_age, to_age):
        """Delete all employees whose age is within [from_age, to_age]. Returns how many were deleted."""
        before = len(self.employees)
        self.employees = [e for e in self.employees if not (from_age <= e.age <= to_age)]
        return before - len(self.employees)

    def find_by_name(self, name):
        """Case-insensitive search. Returns the Employee or None."""
        for emp in self.employees:
            if emp.name.lower() == name.lower():
                return emp
        return None

    def update_salary_by_name(self, name, new_salary):
        emp = self.find_by_name(name)
        if emp is None:
            return False
        emp.salary = new_salary
        return True

### Quick test of the manager (no menu yet)

In [3]:
mgr = EmployeesManager()
mgr.add_employee(Employee("Asha", 28, 55000))
mgr.add_employee(Employee("Ravi", 45, 82000))
mgr.add_employee(Employee("Meena", 33, 61000))
mgr.add_employee(Employee("Karan", 52, 95000))
mgr.list_employees()

print("Find 'meena':", mgr.find_by_name("meena"))
print("Update Ravi:", mgr.update_salary_by_name("Ravi", 90000))
print("Deleted (30-40):", mgr.delete_by_age_range(30, 40))
mgr.list_employees()

+-----------------+-------+--------------+
| Name            |  Age  |       Salary |
+-----------------+-------+--------------+
| Asha            |  28   |    55,000.00 |
| Ravi            |  45   |    82,000.00 |
| Meena           |  33   |    61,000.00 |
| Karan           |  52   |    95,000.00 |
+-----------------+-------+--------------+
Find 'meena': Meena           Age: 33   Salary: 61,000.00
Update Ravi: True
Deleted (30-40): 1
+-----------------+-------+--------------+
| Name            |  Age  |       Salary |
+-----------------+-------+--------------+
| Asha            |  28   |    55,000.00 |
| Ravi            |  45   |    90,000.00 |
| Karan           |  52   |    95,000.00 |
+-----------------+-------+--------------+


## 3. `FrontendManager` class
A menu-driven interface. All input is validated so a typo never crashes the program.

In [4]:
class FrontendManager:
    def __init__(self):
        self.manager = EmployeesManager()

    # ---------- helpers ----------
    def _read_int(self, prompt):
        while True:
            try:
                return int(input(prompt))
            except ValueError:
                print("Please enter a whole number.")

    def _read_float(self, prompt):
        while True:
            try:
                return float(input(prompt))
            except ValueError:
                print("Please enter a valid number.")

    def print_menu(self):
        print("\n===== Employees System =====")
        print("1) Add new employee")
        print("2) List all employees")
        print("3) Delete by age range")
        print("4) Update salary by name")
        print("5) Exit")

    # ---------- actions ----------
    def add_employee(self):
        name = input("Enter name: ").strip()
        age = self._read_int("Enter age: ")
        salary = self._read_float("Enter salary: ")
        self.manager.add_employee(Employee(name, age, salary))
        print(f"Employee '{name}' added.")

    def list_employees(self):
        self.manager.list_employees()

    def delete_by_age(self):
        low = self._read_int("Delete from age: ")
        high = self._read_int("Delete to age: ")
        if low > high:
            low, high = high, low
        count = self.manager.delete_by_age_range(low, high)
        print(f"{count} employee(s) deleted.")

    def update_salary(self):
        name = input("Enter employee name: ").strip()
        salary = self._read_float("Enter new salary: ")
        if self.manager.update_salary_by_name(name, salary):
            print("Salary updated.")
        else:
            print(f"No employee named '{name}' was found.")

    # ---------- main loop ----------
    def run(self):
        actions = {1: self.add_employee, 2: self.list_employees,
                   3: self.delete_by_age, 4: self.update_salary}
        while True:
            self.print_menu()
            choice = self._read_int("Enter your choice (1-5): ")
            if choice == 5:
                print("Goodbye!")
                break
            action = actions.get(choice)
            if action:
                action()
            else:
                print("Invalid choice, please pick 1-5.")

## 4. Demo run with scripted input
The helper replaces `input()` with a list of answers. It simulates: add 3 employees → list → try a bad menu choice → update a salary → delete ages 30-40 → list again → exit.

In [5]:
from unittest.mock import patch

def run_with_inputs(func, inputs):
    it = iter(inputs)
    def fake_input(prompt=""):
        value = next(it)
        print(f"{prompt}{value}")
        return value
    with patch("builtins.input", fake_input):
        return func()

script = [
    "1", "Asha", "28", "55000",
    "1", "Ravi", "forty", "45", "82000",      # 'forty' is rejected, then 45 accepted
    "1", "Meena", "33", "61000",
    "2",
    "9",                                       # invalid menu choice
    "4", "ravi", "90000",
    "4", "nobody", "1000",                     # name not found
    "3", "30", "40",
    "2",
    "5",
]
run_with_inputs(FrontendManager().run, script)


===== Employees System =====
1) Add new employee
2) List all employees
3) Delete by age range
4) Update salary by name
5) Exit
Enter your choice (1-5): 1
Enter name: Asha
Enter age: 28
Enter salary: 55000
Employee 'Asha' added.

===== Employees System =====
1) Add new employee
2) List all employees
3) Delete by age range
4) Update salary by name
5) Exit
Enter your choice (1-5): 1
Enter name: Ravi
Enter age: forty
Please enter a whole number.
Enter age: 45
Enter salary: 82000
Employee 'Ravi' added.

===== Employees System =====
1) Add new employee
2) List all employees
3) Delete by age range
4) Update salary by name
5) Exit
Enter your choice (1-5): 1
Enter name: Meena
Enter age: 33
Enter salary: 61000
Employee 'Meena' added.

===== Employees System =====
1) Add new employee
2) List all employees
3) Delete by age range
4) Update salary by name
5) Exit
Enter your choice (1-5): 2
+-----------------+-------+--------------+
| Name            |  Age  |       Salary |
+-----------------+-----

## 5. Run it interactively
Uncomment the line below to use the menu yourself.

In [6]:
# FrontendManager().run()